# 03 V2 — Live HTML Fetch + Screenshot + Quality Filter

Live-fetches HTML from the candidate URL pool (01_v2 + 02_v2), takes Playwright screenshots,
applies quality filters, and outputs the final `pages_for_generation.jsonl`.

**Why live-fetch instead of WARC extraction?**
- Much simpler — no S3 access, no WARC byte-range requests
- Gets current page content and renderable screenshot
- Gemini is multimodal — the screenshot matters for generation quality

## Pipeline
```
all_candidate_urls.jsonl
    → async HTTP fetch (httpx, 50 concurrent)
    → quality filter (text length, language, not parked/error)
    → Playwright screenshot (headless Chromium)
    → @type classification (URL + keyword heuristics)
    → sample to match type_distribution.json
    → pages_for_generation.jsonl
```

## Output
`data/processed/pages_for_generation.jsonl` — `{url, html, screenshot_path, schema_type}`

## Setup
```bash
pip install httpx[http2] playwright langdetect tqdm
playwright install chromium
```

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import asyncio
import time
import hashlib
import re
from pathlib import Path
from collections import Counter
from dotenv import load_dotenv

load_dotenv('../.env', override=True)

CANDIDATE_PATH = Path('../data/raw/all_candidate_urls.jsonl')
HTML_CACHE_DIR = Path('../data/raw/html_v2')
SCREENSHOT_DIR = Path('../data/screenshots_v2')
OUT_PATH       = Path('../data/processed/pages_for_generation.jsonl')

HTML_CACHE_DIR.mkdir(parents=True, exist_ok=True)
SCREENSHOT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# Load type distribution targets
with open('../config/type_distribution.json') as f:
    TYPE_CONFIG = json.load(f)
TYPE_WEIGHTS = {k: v['weight'] for k, v in TYPE_CONFIG['types'].items()}
TARGET_TOTAL = 10_000  # adjust to budget

print(f'Candidate pool: {CANDIDATE_PATH}')
print(f'Target output:  {TARGET_TOTAL:,} pages')

## 1. Load & Sample Candidates

Load the merged URL pool and sample to get a balanced type distribution before fetching,
so we don't waste HTTP requests on overrepresented types.

In [ ]:
import random

# Load all candidates
candidates = []
with open(CANDIDATE_PATH) as f:
    for line in f:
        candidates.append(json.loads(line))

print(f'Total candidates: {len(candidates):,}')

# Compute per-type target counts (proportional to weight, 3x oversampling)
# We fetch 3x what we need because ~30% of pages will fail quality filter
OVERSAMPLE = 3
total_weight = sum(TYPE_WEIGHTS.values())
type_targets = {
    t: max(100, int(TARGET_TOTAL * OVERSAMPLE * w / total_weight))
    for t, w in TYPE_WEIGHTS.items()
}

# Sample candidates per type
by_type = {}
for r in candidates:
    t = r.get('schema_type', 'LocalBusiness')
    by_type.setdefault(t, []).append(r)

sampled = []
for t, target in type_targets.items():
    pool = by_type.get(t, [])
    n = min(len(pool), target)
    sampled.extend(random.sample(pool, n))
    print(f'  {t:25s} pool={len(pool):5,}  fetching={n:5,}  target={target//OVERSAMPLE:,}')

random.shuffle(sampled)
print(f'\nTotal to fetch: {len(sampled):,}')

## 2. Async HTML Fetch

Fetches HTML with a 10s timeout and 50 concurrent connections.
Results cached to disk — safe to interrupt and resume.

In [ ]:
import httpx
from tqdm.notebook import tqdm

FETCH_CONCURRENCY = 50
FETCH_TIMEOUT     = 10   # seconds
MIN_HTML_CHARS    = 500
MAX_HTML_CHARS    = 500_000

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (compatible; SchemaBot/1.0; research)',
    'Accept': 'text/html,application/xhtml+xml',
    'Accept-Language': 'en-US,en;q=0.9',
}

def url_to_cache_path(url: str) -> Path:
    key = hashlib.md5(url.encode()).hexdigest()
    return HTML_CACHE_DIR / f'{key}.html'

async def fetch_one(client: httpx.AsyncClient, record: dict, sem: asyncio.Semaphore) -> dict | None:
    url = record['url']
    cache = url_to_cache_path(url)

    # Cache hit
    if cache.exists():
        html = cache.read_text(errors='replace')
        return {**record, 'html': html, 'cached': True}

    async with sem:
        try:
            resp = await client.get(url, timeout=FETCH_TIMEOUT, follow_redirects=True)
            if resp.status_code != 200:
                return None
            ct = resp.headers.get('content-type', '')
            if 'html' not in ct and 'text' not in ct:
                return None
            html = resp.text
            if len(html) < MIN_HTML_CHARS:
                return None
            # Trim very large pages
            html = html[:MAX_HTML_CHARS]
            cache.write_text(html)
            return {**record, 'html': html, 'cached': False}
        except Exception:
            return None

async def fetch_all(records: list[dict]) -> list[dict]:
    sem = asyncio.Semaphore(FETCH_CONCURRENCY)
    results = []
    async with httpx.AsyncClient(headers=HEADERS) as client:
        tasks = [fetch_one(client, r, sem) for r in records]
        for coro in tqdm(asyncio.as_completed(tasks), total=len(tasks), desc='Fetching'):
            result = await coro
            if result:
                results.append(result)
    return results

print(f'Fetching {len(sampled):,} URLs ({FETCH_CONCURRENCY} concurrent)...')
fetched = await fetch_all(sampled)

cached  = sum(1 for r in fetched if r.get('cached'))
print(f'\nFetched: {len(fetched):,} OK  ({cached:,} from cache, {len(sampled)-len(fetched):,} failed)')

## 3. Quality Filter

Filter fetched pages for:
- Enough text content (real business page vs. parked domain)
- English language
- Not an error/maintenance page
- Has business identity signals

In [ ]:
from html.parser import HTMLParser

try:
    from langdetect import detect, LangDetectException
    LANGDETECT = True
except ImportError:
    LANGDETECT = False
    print('langdetect not installed — skipping language detection')

# Patterns that indicate a parked/thin/error page
PARKED_PATTERNS = re.compile(
    r'domain (is |for sale|parking)|coming soon|under construction|'
    r'website coming|buy this domain|this site is for sale|'
    r'404 not found|403 forbidden|access denied',
    re.IGNORECASE
)

def extract_text(html: str) -> str:
    """Simple text extraction from HTML."""
    # Remove scripts and styles
    html = re.sub(r'<(script|style)[^>]*>.*?</(script|style)>', '', html, flags=re.DOTALL | re.IGNORECASE)
    # Remove tags
    text = re.sub(r'<[^>]+>', ' ', html)
    # Collapse whitespace
    return re.sub(r'\s+', ' ', text).strip()

def quality_filter(record: dict) -> bool:
    html = record.get('html', '')
    text = extract_text(html)

    # Minimum text length (real pages have content)
    if len(text) < 300:
        return False

    # Parked / error page check
    if PARKED_PATTERNS.search(text[:2000]):
        return False

    # Language detection (English only)
    if LANGDETECT:
        try:
            lang = detect(text[:3000])
            if lang != 'en':
                return False
        except Exception:
            pass  # if detection fails, keep the page

    return True

filtered = [r for r in fetched if quality_filter(r)]

print(f'After quality filter: {len(filtered):,} / {len(fetched):,} pages')
print(f'Filter pass rate: {len(filtered)/len(fetched)*100:.1f}%')

## 4. Screenshot with Playwright

Takes a full-page screenshot of each quality-filtered page.
Screenshots cached to disk — skips already-rendered pages.

In [ ]:
from playwright.async_api import async_playwright

SCREENSHOT_WIDTH  = 1280
SCREENSHOT_HEIGHT = 900
SCREENSHOT_TIMEOUT = 15_000  # ms

def url_to_screenshot_path(url: str) -> Path:
    key = hashlib.md5(url.encode()).hexdigest()
    return SCREENSHOT_DIR / f'{key}.png'

async def screenshot_batch(records: list[dict]) -> list[dict]:
    results = []
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(
            viewport={'width': SCREENSHOT_WIDTH, 'height': SCREENSHOT_HEIGHT},
            user_agent='Mozilla/5.0 (compatible; SchemaBot/1.0; research)',
        )

        for record in tqdm(records, desc='Screenshots'):
            url  = record['url']
            path = url_to_screenshot_path(url)

            if path.exists():
                results.append({**record, 'screenshot_path': str(path)})
                continue

            page = await context.new_page()
            try:
                await page.goto(url, timeout=SCREENSHOT_TIMEOUT, wait_until='domcontentloaded')
                await page.screenshot(path=str(path), full_page=False)
                results.append({**record, 'screenshot_path': str(path)})
            except Exception:
                pass  # screenshot failed — skip this page
            finally:
                await page.close()

        await browser.close()
    return results

print(f'Taking screenshots for {len(filtered):,} pages...')
with_screenshots = await screenshot_batch(filtered)

print(f'\nScreenshots OK: {len(with_screenshots):,} / {len(filtered):,}')

## 5. Sample to Target Distribution

Sample the quality-filtered + screenshotted pages to match `type_distribution.json`.

In [ ]:
# Compute per-type final targets
final_targets = {
    t: max(50, int(TARGET_TOTAL * w / total_weight))
    for t, w in TYPE_WEIGHTS.items()
}

by_type_final = {}
for r in with_screenshots:
    t = r.get('schema_type', 'LocalBusiness')
    by_type_final.setdefault(t, []).append(r)

final_pages = []
print('Final type sampling:')
for t, target in final_targets.items():
    pool = by_type_final.get(t, [])
    n = min(len(pool), target)
    final_pages.extend(random.sample(pool, n) if n < len(pool) else pool)
    print(f'  {t:25s} have={len(pool):5,}  taking={n:4,}  target={target:4,}')

# Include any types not in config (don't waste them)
configured_types = set(TYPE_WEIGHTS.keys())
for t, pages in by_type_final.items():
    if t not in configured_types:
        extra = pages[:200]
        final_pages.extend(extra)
        print(f'  {t:25s} (unconfigured) taking={len(extra)}')

random.shuffle(final_pages)
print(f'\nFinal page count: {len(final_pages):,}')

## 6. Save Output

In [ ]:
# Save pages_for_generation.jsonl
# Drop the full HTML from the index record — store path only for large HTMLs,
# but keep HTML inline since Gemini needs it and files are 03_v2's output
saved = 0
with open(OUT_PATH, 'w') as f:
    for page in final_pages:
        record = {
            'url':             page['url'],
            'schema_type':     page.get('schema_type', 'LocalBusiness'),
            'html':            page.get('html', ''),
            'screenshot_path': page.get('screenshot_path', ''),
            'source':          page.get('source', 'unknown'),
        }
        f.write(json.dumps(record, ensure_ascii=False) + '\n')
        saved += 1

size_mb = OUT_PATH.stat().st_size / 1e6
print(f'Saved {saved:,} pages → {OUT_PATH} ({size_mb:.1f} MB)')

## 7. Summary

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from io import BytesIO
from IPython.display import Image as IPImage

type_dist  = Counter(p['schema_type'] for p in final_pages)
source_dist = Counter(p.get('source', 'unknown') for p in final_pages)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('pages_for_generation.jsonl — composition', fontsize=13, fontweight='bold')

labels, counts = zip(*type_dist.most_common(15))
ax1.barh(range(len(labels)), counts, color='steelblue')
ax1.set_yticks(range(len(labels)))
ax1.set_yticklabels(labels, fontsize=9)
ax1.set_xlabel('Pages')
ax1.set_title('@type distribution')
ax1.invert_yaxis()

src_labels, src_counts = zip(*source_dist.most_common())
ax2.pie(src_counts, labels=src_labels, autopct='%1.0f%%', startangle=90)
ax2.set_title('Source (WDC vs CC)')

plt.tight_layout()
buf = BytesIO()
fig.savefig(buf, format='png', dpi=150, bbox_inches='tight')
plt.close()
buf.seek(0)
display(IPImage(data=buf.getvalue()))

print(f'\nTotal pages: {len(final_pages):,}')
print(f'Ready for: 06_v2_synthetic_generation.ipynb')